# F02 — Ghostbuster features with Fast-DetectGPT (GPT-Neo-2.7B), stage S2

Runtime: **GPU (T4)**, about 30–40 minutes for 9,000 documents. Scores are cached in
`fdgpt_cache.csv` every 250 documents, so a disconnected session can resume by running all
cells again. Output: `ghostbuster_fdgpt.parquet` + manifest. Outcomes are not recomputed
here; they are joined from the TF-IDF feature table by `doc_id`.

In [ ]:
# Fresh runtime recommended (Runtime > Disconnect and delete runtime) if d2vi was installed before.
!pip -q install vaderSentiment textstat nrclex transformers accelerate huggingface_hub
!pip -q install --force-reinstall --no-deps "d2vi @ git+https://github.com/DRuanli/dvai"
import nltk
nltk.download("cmudict", quiet=True); nltk.download("subjectivity", quiet=True)
SUBJ_DIR = nltk.data.find("corpora/subjectivity")
import torch, d2vi
print("d2vi", d2vi.__version__, "| GPU:", torch.cuda.is_available())
assert d2vi.__version__ >= "2.0.0.dev1", "old d2vi still loaded: restart the runtime and run all"

In [ ]:
import os
if not os.path.exists("gb"):
    !curl -sL -o gb.zip https://codeload.github.com/vivek3141/ghostbuster-data/zip/refs/heads/master
    !unzip -qq gb.zip 'ghostbuster-data-master/essay/*' 'ghostbuster-data-master/reuter/*' 'ghostbuster-data-master/wp/*' -d .
    !mv ghostbuster-data-master gb
from d2vi.data.loaders import load_ghostbuster
docs = load_ghostbuster("gb")
print(docs.groupby(["domain", "source"]).size())

In [ ]:
import numpy as np, pandas as pd
from d2vi.features.neural import FastDetectGPT
from d2vi.features.outcomes import split_halves
CACHE = "fdgpt_cache.csv"
done = pd.read_csv(CACHE) if os.path.exists(CACHE) else pd.DataFrame(columns=["doc_id", "score_raw", "score_raw_h1"])
todo = docs[~docs["doc_id"].isin(done["doc_id"])]
print("cached", len(done), "to score", len(todo))
if len(todo):
    det = FastDetectGPT()
    buf = []
    for i, (doc_id, text) in enumerate(zip(todo["doc_id"], todo["text"])):
        buf.append({"doc_id": doc_id, "score_raw": det.score_one(text),
                    "score_raw_h1": det.score_one(split_halves(text)[0])})
        if len(buf) == 250 or i == len(todo) - 1:
            done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True); done.to_csv(CACHE, index=False); buf = []
            print(len(done), "scored")

In [ ]:
from d2vi.features.pipeline import save_feature_table, package_versions
feats = docs[["doc_id"]].merge(done, on="doc_id", how="left").drop(columns="doc_id")
feats["length"] = [len(t.split()) for t in docs["text"]]
manifest = {"corpus": "ghostbuster-data (codeload zip, master)", "detector": "Fast-DetectGPT analytic, EleutherAI/gpt-neo-2.7B, 400 tokens",
            "threshold": "computed downstream: 95th percentile of human scores in the other domains (v1 rule)",
            "versions": package_versions("d2vi", "transformers", "torch")}
path = save_feature_table(docs, feats, "ghostbuster_fdgpt.parquet", manifest)
from google.colab import files
files.download(str(path)); files.download(str(path.with_suffix(".manifest.json")))